---

In [41]:
library(AnnotationDbi)
library(org.Hs.eg.db)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(dplyr)
library(rtracklayer)

## Transcript -level genebody

##### first up regulated

In [42]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/upregulated_genes_deseq.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [43]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [44]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
mygenes.transcripts <- subset(
  transcripts(txdb, columns = c("tx_id", "tx_name", "gene_id")),
  gene_id %in% entrez_ids
)

In [45]:
gene_bodies <- as.data.frame(mygenes.transcripts)

In [46]:
#as.data.frame(gene_bodies)

In [47]:
library(dplyr)

# This regex matches only chr1-chr22, chrX, chrY, chrM (no underscores or suffixes)
gene_bodies <- gene_bodies %>%
  filter(grepl("^chr([1-9]|1[0-9]|2[0-2]|X|Y|M)$", seqnames))

In [48]:
as.data.frame(gene_bodies)

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,11934205,11958620,24416,+,955,ENST00000485046.5,5351
chr1,11934694,11952691,17998,+,956,ENST00000449038.5,5351
chr1,11934717,11975537,40821,+,957,ENST00000196061.5,5351
chr1,11934734,11954767,20034,+,958,ENST00000358133.5,5351
chr1,11934743,11957943,23201,+,959,ENST00000429000.6,5351
chr1,11954301,11958593,4293,+,960,ENST00000465920.1,5351
chr1,11964688,11975538,10851,+,961,ENST00000491536.5,5351
chr1,11964702,11966414,1713,+,962,ENST00000470133.1,5351
chr1,11971543,11975536,3994,+,963,ENST00000481933.1,5351


In [49]:
# Converting to dataframe
tss_df <- as.data.frame(gene_bodies)

# Creating a .bed like table
bed_gene_df <- data.frame(
  chrom = tss_df$seqnames,
  chromStart = tss_df$start - 1,  # BED is 0-based
  chromEnd = tss_df$end,          
  name = tss_df$tx_name,         
  score = 0,
  strand = tss_df$strand
)
#export
write.table(
  bed_gene_df,
  file = "upregulated_transcript_level_genebodies.bed",
  sep = "\t",
  quote = FALSE,
  row.names = FALSE,
  col.names = FALSE
)


In [50]:
#tss_df

In [51]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<dbl>,<int>,<chr>,<dbl>,<fct>
chr1,11934204,11958620,ENST00000485046.5,0,+
chr1,11934693,11952691,ENST00000449038.5,0,+
chr1,11934716,11975537,ENST00000196061.5,0,+
chr1,11934733,11954767,ENST00000358133.5,0,+
chr1,11934742,11957943,ENST00000429000.6,0,+
chr1,11954300,11958593,ENST00000465920.1,0,+
chr1,11964687,11975538,ENST00000491536.5,0,+
chr1,11964701,11966414,ENST00000470133.1,0,+
chr1,11971542,11975536,ENST00000481933.1,0,+


##### now downregulated

In [52]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/downregulated_genes_deseq.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [53]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [54]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
mygenes.transcripts <- subset(
  transcripts(txdb, columns = c("tx_id", "tx_name", "gene_id")),
  gene_id %in% entrez_ids
)

In [55]:
gene_bodies <- as.data.frame(mygenes.transcripts)

In [56]:
#as.data.frame(gene_bodies)

In [57]:
library(dplyr)

# This regex matches only chr1-chr22, chrX, chrY, chrM (no underscores or suffixes)
gene_bodies <- gene_bodies %>%
  filter(grepl("^chr([1-9]|1[0-9]|2[0-2]|X|Y|M)$", seqnames))

In [58]:
as.data.frame(gene_bodies)

seqnames,start,end,width,strand,tx_id,tx_name,gene_id
<fct>,<int>,<int>,<int>,<fct>,<int>,<chr>,<list>
chr1,960584,965719,5136,+,130,ENST00000338591.8,339451
chr1,961449,962478,1030,+,131,ENST00000463212.1,339451
chr1,962727,964530,1804,+,132,ENST00000466300.1,339451
chr1,963552,964164,613,+,133,ENST00000481067.1,339451
chr1,1615454,1630604,15151,+,245,ENST00000479659.5,142678
chr1,1615496,1630604,15109,+,246,ENST00000489635.5,142678
chr1,1615500,1630605,15106,+,247,ENST00000355826.10,142678
chr1,1615500,1630605,15106,+,248,ENST00000505820.7,142678
chr1,1615500,1630605,15106,+,249,ENST00000518681.6,142678


In [59]:
# Converting to dataframe
tss_df <- as.data.frame(gene_bodies)

# Creating a .bed like table
bed_gene_df <- data.frame(
  chrom = tss_df$seqnames,
  chromStart = tss_df$start - 1,  # BED is 0-based
  chromEnd = tss_df$end,          
  name = tss_df$tx_name,         
  score = 0,
  strand = tss_df$strand
)
#export
write.table(
  bed_gene_df,
  file = "downregulated_transcript_level_genebodies.bed",
  sep = "\t",
  quote = FALSE,
  row.names = FALSE,
  col.names = FALSE
)


In [60]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<dbl>,<int>,<chr>,<dbl>,<fct>
chr1,960583,965719,ENST00000338591.8,0,+
chr1,961448,962478,ENST00000463212.1,0,+
chr1,962726,964530,ENST00000466300.1,0,+
chr1,963551,964164,ENST00000481067.1,0,+
chr1,1615453,1630604,ENST00000479659.5,0,+
chr1,1615495,1630604,ENST00000489635.5,0,+
chr1,1615499,1630605,ENST00000355826.10,0,+
chr1,1615499,1630605,ENST00000505820.7,0,+
chr1,1615499,1630605,ENST00000518681.6,0,+
